# NJ Pharma Supply Chain ESG — Risk Scoring Methodology

**Author:** Yiduo Xiao · Lunarix Technologies LLC · `yijimu@lunariduo.com`

This notebook documents the TCFD-aligned **5×5 likelihood × impact** scoring methodology applied to the supply-chain ESG risks of six NJ-headquartered pharma manufacturers (J&J, Merck, BMS, Bayer, Sanofi, BD).

**Inputs:** `data/risk_matrix.csv` (30 cells = 6 stages × 5 risk types) · `data/pillar_scores.csv` · `data/emissions_5yr.csv`

**Outputs:** Composite ESG score per company · Top 3 hotspots · Scope 3 share computation · paired risk/opportunity register.

## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 2. Load the risk matrix

In [ ]:
risk = pd.read_csv('../data/risk_matrix.csv')
risk['inherent_score'] = risk['likelihood'] * risk['impact']
print(f'Risk cells loaded: {len(risk)}')
risk.head(10)

## 3. Identify the top hotspots (score ≥ 18)

These are the supply-chain stages × risk-type combinations where mitigation has the highest expected payoff.

In [ ]:
hotspots = risk[risk['inherent_score'] >= 18].sort_values('inherent_score', ascending=False)
print(f'Critical hotspots (score ≥ 18): {len(hotspots)}')
hotspots[['stage', 'risk_type', 'likelihood', 'impact', 'inherent_score']]

## 4. Heat map visualization

Stages on the x-axis, risk types on the y-axis. Color = inherent score.

In [ ]:
pivot = risk.pivot(index='risk_type', columns='stage', values='inherent_score')
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(pivot.values, cmap='RdYlGn_r', vmin=0, vmax=25, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=20, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, int(val), ha='center', va='center', color='white' if val >= 15 else 'black', fontweight='bold')
ax.set_title('Supply Chain Risk Heat Map · likelihood × impact', loc='left', fontweight='bold')
plt.colorbar(im, ax=ax, label='Inherent risk score (1–25)')
plt.tight_layout()
plt.show()

## 5. Composite ESG score per company (SASB pharma materiality weights)

Weights: **E 40% · S 35% · G 25%** — reflecting SASB Pharmaceutical Standard materiality for the industry.

In [ ]:
pillar = pd.read_csv('../data/pillar_scores.csv')
pillar['composite_check'] = 0.40 * pillar['pillar_e'] + 0.35 * pillar['pillar_s'] + 0.25 * pillar['pillar_g']
pillar[['ticker', 'pillar_e', 'pillar_s', 'pillar_g', 'composite', 'composite_check']]

## 6. Scope 3 share — the strategic carbon opportunity

Computing what % of each company's emissions sit upstream (Scope 3). Industry pattern: ~80% of pharma carbon footprint is Scope 3.

In [ ]:
emissions = pd.read_csv('../data/emissions_5yr.csv')
latest_year = emissions['reporting_year'].max()
latest = emissions[emissions['reporting_year'] == latest_year]
pivot_em = latest.pivot(index='ticker', columns='scope', values='co2e_mt_million')
pivot_em['total'] = pivot_em.sum(axis=1)
pivot_em['scope3_pct'] = (pivot_em[3] / pivot_em['total']) * 100
print(f'\nNJ pharma cluster avg Scope 3 share (year {latest_year}): {pivot_em["scope3_pct"].mean():.1f}%')
pivot_em.sort_values('scope3_pct', ascending=False)

## 7. Risk → Opportunity register (paired output)

Each material risk gets a paired business opportunity and a recommended next-step action. This is the artifact that goes to the sustainability lead or supply-chain analyst for quarterly planning.

In [ ]:
register = pd.DataFrame([
    {'risk': 'Scope 3 emissions concentration in API sourcing',
     'tcfd_class': 'Transition',
     'opportunity': 'SBTi-aligned supplier engagement',
     'quantified_value': '−14% Scope 3 by 2030',
     'action': 'Issue CDP Supply Chain disclosure request to top 50 API suppliers'},
    {'risk': 'Cold chain disruption — NJ flood corridor',
     'tcfd_class': 'Physical',
     'opportunity': 'IoT-monitored cold chain + redundant routing',
     'quantified_value': '−22% spoilage rate',
     'action': 'Cross-ref cold-storage locations against FEMA flood zones; retrofit 3 sites'},
    {'risk': 'API single-source (CN/IN) tariff exposure',
     'tcfd_class': 'Transition',
     'opportunity': 'Nearshoring + IRA Section 60101 credits',
     'quantified_value': '$40–60M working capital release',
     'action': 'TCO model for top 20 single-sourced molecules'},
    {'risk': 'Data integrity in supplier audit trail',
     'tcfd_class': 'Regulatory',
     'opportunity': 'GCP BigQuery + ALCOA+ compliant pipeline',
     'quantified_value': '40% audit prep time reduction',
     'action': 'Pilot data-contract intake for 5 paper-based suppliers'},
])
register

## Limitations

- All data is from **public sources**. Composite scores are my synthesis and will not match any single rating provider exactly.
- Supplier-level granularity is not publicly available; cluster aggregate only.
- Cost projections ($80–140M, $40–60M) are scenario-based, not forecasts.